# CSI-FiSh amb $p=419$

En aquest quadern implementarem l'esquema de firma digital CSI-FiSh per al primer $p=419$ i donarem alguns exemples. Només hem considerat aquest cas perquè el càlcul de l'acció depèn molt del primer escollit, ja que s'ha de conèixer l'estructura del grup de classes d'ideals.

Pel cas $p=419$, ja sabem que $N=27$ i $\mathfrak{l}_1$ és un generador del grup de classes d'ideals. A més, en el quadern `CSIDH.ipynb` ja hem calculat l'acció de $\mathfrak{l}_1$ sobre totes les corbes de $\mathcal{E}_{\mathbb{F}_p}$ (Taula 4.1 de la memòria).

Aleshores, per simplificar el quadern i centrar-nos en els algoritmes de CSI-FiSh, utilitzarem l'acció precalculada en comptes d'utilitzar l'Algoritme 1 de la memòria. Notem que en els casos petits sempre es pot procedir així, ja que és factible precalcular els cicles que generen els ideals $\mathfrak{l}_i$. En canvi, per casos més grans (com els paràmetres de CSIDH-512), s'haurien d'utilitzar tècniques més sofisticades, tal com hem explicat a la memòria.

## Implementació
En aquest secció implementarem CSI-FiSh per $p=419$.

Primer, importarem el mòdul `hashlib` (emprarem `SHAKE256`per la funció resum) i definirem els paràmetres.

In [1]:
import hashlib

# Paràmetres
p  = 419  # 4*3*5*7-1
N  = 27   # #Cl(O)

Ara, donam la funció per calcular l'acció de Waterhouse per aquest cas particular. En comptes de treballar amb ECs, treballarem directament sobre els coeficients de Montgomery. Hem afegit el paràmetre opcional `twist`, que en cas de ser `True` indica a la funció que s'ha de calcular l'acció a partir del twist quadràtic de `pub` (ens serà útil per la verificació de les signatures).

In [2]:
# Calcula l'acció de Waterhouse en el cas de CSI-FiSh amb p=419
def accio(pub, priv, twist=False):
    # Cicle que genera l1 (Taula 4.1 de la memòria)
    cicle = [
        0,   158, 410, 368, 404,  75, 144, 191, 174,
        413, 379, 124, 199, 390,  29, 220, 295, 40,
        6,   245, 228, 275, 344,  15,  51,  9,  261
    ]

    # Partint de pub (o del twist quadatic) ens movem priv vegades en el cicle
    index_orig = cicle.index(p - pub if twist else pub)
    index_dest = (index_orig + priv) % N
    return cicle[index_dest]

El primer algoritme de CSI-FiSh és la generació de claus (Algoritme 2 de la memòria). De manera anàloga al quadern `CSIDH.ipynb`, aquí fixarem les claus privades per simplificar, però aquestes s'haurien de generar aleatòriament (i de manera uniforme).

In [3]:
# Genera les claus públiques a partir de les privades.
def generar_pub(privs):
    # Aplicam l'acció per cadascuna de les claus privades.
    return [accio(0, priv) for priv in privs]

El segon algoritme és la firma (Algoritme 3 de la memòria). Ara bé, aquest algoritme es pot dividir clarament en les tres passes del protocol Sigma: compromisos, reptes, i respostes. A continuació donam una funció per cadascuna de les passes.

In [4]:
# Genera els compromisos (ui, Ui=[ui]E0, per i=1..t)
def gen_compromisos(t):
    # Generam u1,...,ut aleatòriament i calculam el coef. de Montgomery de [ui]E_0
    u = [randint(0, N-1) for _ in range(t)]
    compromisos = [accio(0, ui) for ui in u]

    print("u:          ", u)
    print("Compromisos:", compromisos, "(Coeficients de Montgomery de [ui]E_0)")

    return u, compromisos

In [5]:
# Obté els reptes, aplicant H(U1||...||Ut||msg)
def gen_reptes(compromisos, msg, S):
    # Paràmetre t
    t = len(compromisos)

    # Empram shake256, que ens permet adaptar la longitud del resum
    c = hashlib.shake_256()
    c.update(b"CSI-FISH|reptes|")

    # Concatenam els compromisos
    for compromis in compromisos:
        # Representam les corbes amb el coeficient de Montgomery (p=419 -> 2 bytes)
        c.update(int(compromis).to_bytes(2, byteorder="big"))

    # Concatenam el missatge
    c.update(msg.encode())

    # Obtenim el resum (t bytes).
    #
    # Nota. En teoria voldríem t*(s+1) bits, però és més còmode treballar amb
    # bytes. Si restringim 2<=S<=128 (1<=s<=7 bits), podem representar els
    # reptes amb s+1<=8 bits, ja que se li ha d'afegir el signe. Per tant, podem
    # prendre un byte per cada repte i, llavors, obtenir el signe i fer mòdul S.
    resum = c.digest(t)

    # Obtenim un repte per cada byte (el signe el proporcionarà el bit més significatiu, i llavors fem mòdul S)
    reptes = [-((r >> 7)*2-1) * (r % S) for r in resum]

    # Mostram el resum i els reptes
    print("Resum: ", resum)
    print("Reptes:", reptes)

    # Retornam els reptes
    return reptes


In [6]:
# Obté les respostes als reptes
def gen_respostes(u, reptes, privs):
    # Paràmetre t
    t = len(u)

    # Calculam les respostes (wi = vi - sign(vi) * priv[|vi|])
    respostes = [(u[i] - sign(reptes[i]) * privs[abs(reptes[i])]) % N for i in range(t)]

    print("Respostes:", respostes)
    return respostes

Ara, ajuntam les tres funcions anteriors per finalitzar l'algoritme de firma.

In [7]:
# Firma un missage, aplicant una clau privada i t rondes
def firmar(msg, privs, t):
    # Paràmetre S
    S = len(privs)

    # 1. COMPROMISOS
    print("1. COMPROMISOS")
    u, compromisos = gen_compromisos(t)

    # 2. REPTES
    print("\n2. REPTES")
    reptes = gen_reptes(compromisos, msg, S)

    # 3. RESPOSTES
    print("\n3. RESPOSTES")
    respostes = gen_respostes(u, reptes, privs)

    # SIGNATURA
    # 
    # Nota. En realitat hauria de ser una tira de bits compacta.
    # En aquesta implementació retornarem un diccionari per fer-ho més llegible.
    return {
        "reptes": reptes,
        "respostes": respostes,
    }

El tercer i darrer algoritme de CSI-FiSh és la verificació (Algoritme 4 de la memòria). Donada una signatura per un missatge i una clau pública, el verificador ha de recalcular els compromisos (notem que aquí utilitzarem el paràmetre `twist` de la funció `accio`) i, després, verificar que els reptes s'hagin generat correctament.

In [8]:
# Verifica una signatura
def verificar(msg, pubs, signatura):
    # Parts de la signatura
    print("Signatura:", signatura)
    reptes = signatura["reptes"]
    respostes = signatura["respostes"]

    # Paràmetres
    S, t = len(pubs), len(reptes)

    # Calculam els compromisos associats
    # (Haurien de ser iguals als compromisos que han generat la signatura)
    compromisos = [accio(pubs[abs(reptes[i])], respostes[i], sign(reptes[i]) == -1) for i in range(t)]
    print("Compromisos associats: ", compromisos)

    # Verificam si el firmant ha generat els reptes correctament
    valida = gen_reptes(compromisos, msg, S) == reptes
    if valida:
        print("Firma vàlida :)")
    else:
        print("Firma no vàlida :(")

    return valida

## Exemple bàsic

El primer exemple que donarem és utilitzant un sol parell de claus ($S=2$) i una sola ronda ($t=1$).

Primer, fixarem els paràmetres i el missatge:

In [9]:
msg  = "PEIX"
priv = [0, 12]
t    = 1

Ara, generarem la clau pública:

In [10]:
pub = generar_pub(priv)
print("Clau privada:", priv)
print("Clau pública:", pub)

Clau privada: [0, 12]
Clau pública: [0, 199]


Aleshores, calcularem una signatura per aquest missatge i aquest parell de claus. Fixarem una llavor d'aleatorietat per tal que els compromisos que es generin al quadern sempre siguin els mateixos.

In [11]:
set_random_seed(20)

signatura = firmar(msg, priv, t)

print("\nSIGNATURA\n", signatura)

1. COMPROMISOS
u:           [13]
Compromisos: [390] (Coeficients de Montgomery de [ui]E_0)

2. REPTES
Resum:  b'\xe7'
Reptes: [-1]

3. RESPOSTES
Respostes: [25]

SIGNATURA
 {'reptes': [-1], 'respostes': [25]}


Per generar aquesta signatura s'ha seguit el procediment següent:
1. Per generar els compromisos, s'ha triat aleatòriament $u_1=13$, que es correspon amb $U_1=E_{390}$.
2. Després, s'ha obtingut el repte calculant el resum de $U_1||msg$, obtenint $v_1=-1$.
3. La resposta al repte és $w_1=u_1-signe(v_1)x_{|v_1|}=13+12=25$.
4. Finalment, la signatura és el conjunt ordenat de reptes i respostes.

Ara, verificam que aquesta signatura és vàlida per aquell missatge i aquell parell de claus concrets.

In [12]:
verificar(msg, pub, signatura)

Signatura: {'reptes': [-1], 'respostes': [25]}
Compromisos associats:  [390]
Resum:  b'\xe7'
Reptes: [-1]
Firma vàlida :)


True

A continuació farem algunes alteracions per veure si la signatura segueix essent vàlida o no.

Per exemple, vegem que si el missatge es canvia a "PEIXOS", la signatura deixa de ser vàlida:

In [13]:
verificar("PEIXOS", pub, signatura)

Signatura: {'reptes': [-1], 'respostes': [25]}
Compromisos associats:  [390]
Resum:  b'p'
Reptes: [0]
Firma no vàlida :(


False

També podem provar canviant el parell de claus:

In [14]:
verificar(msg, generar_pub([0, 11]), signatura) 

Signatura: {'reptes': [-1], 'respostes': [25]}
Compromisos associats:  [29]
Resum:  b'\xb0'
Reptes: [0]
Firma no vàlida :(


False

O fins i tot la pròpia signatura:

In [15]:
sig2 = signatura.copy()
sig2["reptes"][0] = 0
verificar(msg, pub, sig2)

Signatura: {'reptes': [0], 'respostes': [25]}
Compromisos associats:  [9]
Resum:  b'\xef'
Reptes: [-1]
Firma no vàlida :(


False

Ara bé, també és molt fàcil rompre aquest esquema de jugueta. Per exemple, aqusta mateixa signatura també és vàlida per al missatge "Balena".

In [16]:
verificar("Balena", pub, signatura)

Signatura: {'reptes': [0], 'respostes': [25]}
Compromisos associats:  [9]
Resum:  b'\xe0'
Reptes: [0]
Firma vàlida :)


True

## Exemple amb $S=4$ i $t=3$

Ara donarem un exemple un poc més complet, amb $S=2^2=4$ i $t=3$. El procediment és completament anàleg a l'exemple anterior.

In [17]:
msg  = "PEIX"
priv = [0, 18, 08, 26]
t    = 3

In [18]:
pub = generar_pub(priv)
print("Clau privada:", priv)
print("Clau pública:", pub)

Clau privada: [0, 18, 8, 26]
Clau pública: [0, 6, 174, 261]


In [19]:
set_random_seed(1728)

signatura = firmar(msg, priv, t)

print("\nSIGNATURA\n", signatura)

1. COMPROMISOS
u:           [17, 16, 21]
Compromisos: [40, 295, 275] (Coeficients de Montgomery de [ui]E_0)

2. REPTES
Resum:  b'\xa7\xfbY'
Reptes: [-3, -3, 1]

3. RESPOSTES
Respostes: [16, 15, 3]

SIGNATURA
 {'reptes': [-3, -3, 1], 'respostes': [16, 15, 3]}


Com podem veure, el procediment és el mateix que a l'exemple anterior, només que ara es generen 3 compromisos, 3 reptes (ara en el rang $\{-3,\ldots,3\}$, ja que $S=4$), i 3 respostes. En aquest cas és convenient verificar que els 3 compromisos siguin diferents, per evitar donar una solució de MT-GAIP. En un cas real no seria necessari, ja que la probabilitat que això succeeixi és negligible.

Finalment, verifiquem la signatura:

In [20]:
verificar(msg, pub, signatura)

Signatura: {'reptes': [-3, -3, 1], 'respostes': [16, 15, 3]}
Compromisos associats:  [40, 295, 275]
Resum:  b'\xa7\xfbY'
Reptes: [-3, -3, 1]
Firma vàlida :)


True

En aquest exemple també podríem provar d'alterar la signatura, però no aportaríem res de nou respecte l'exemple anterior. Tampoc no té sentit donar un exemple més "real", ja que només hem implementat l'esquema per $p=419$.